# Session 11 · Build a stream, then break it

Crown Street Markets, a fictional 40-store Charlotte grocery chain, streams every register sale into BigQuery for the store managers' staffing dashboard.

In [ ]:
source ~/dsba6190-live-demo-11/env.sh && echo "$TOPIC" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

In [ ]:
n () { bq --project_id="$PROJECT" --quiet query --use_legacy_sql=false --nouse_cache --format=csv "$1" | tail -1 | grep -E "^[0-9]+$" || echo 0; }

## Step 1 · The substrate, in ninety seconds

In [ ]:
gcloud pubsub topics create "$SCRATCH" --project "$PROJECT"
gcloud pubsub subscriptions create "$SCRATCH" --topic "$SCRATCH" --project "$PROJECT"

In [ ]:
for i in 1 2 3; do gcloud pubsub topics publish "$SCRATCH" --project "$PROJECT" --message "register test $i" --attribute store_id=CLT-00$i; done

In [ ]:
gcloud pubsub subscriptions pull "$SCRATCH" --project "$PROJECT" --auto-ack --limit 3 \
  --format="table(message.data.decode(base64),message.attributes.store_id,message.messageId)"

## Step 2 · Three jobs, started before class

In [ ]:
gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name~crown-.*-$SUFFIX" \
  --format="table(name,state,creationTime)"

In [ ]:
for v in baseline lateness dedup; do
  ID=$(gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name=crown-$v-$SUFFIX" --format="value(id)")
  echo "https://console.cloud.google.com/dataflow/jobs/$REGION/$ID?project=$PROJECT"
done

## Step 3 · A burst, and rows in BigQuery

In [ ]:
BURST_AT=$(date -u +%Y-%m-%dT%H:%M:%SZ)
(cd pipeline && $PY registers.py "$T/$TOPIC" burst 200)

In [ ]:
until [ "$(n "SELECT SUM(sales) FROM \`$PROJECT.$DATASET.sales_baseline\` WHERE window_start >= TIMESTAMP_TRUNC('$BURST_AT', MINUTE)")" -ge 200 ] \
   && [ "$(n "SELECT SUM(sales) FROM \`$PROJECT.$DATASET.sales_dedup\` WHERE window_start >= TIMESTAMP_TRUNC('$BURST_AT', MINUTE)")" -ge 200 ]; do
  sleep 15
done
q "SELECT 'baseline' job, SUM(sales) sales, ROUND(SUM(revenue),2) revenue, COUNT(DISTINCT window_start) windows FROM \`$PROJECT.$DATASET.sales_baseline\` WHERE window_start >= TIMESTAMP_TRUNC('$BURST_AT', MINUTE)
   UNION ALL SELECT 'dedup', SUM(sales), ROUND(SUM(revenue),2), COUNT(DISTINCT window_start) FROM \`$PROJECT.$DATASET.sales_dedup\` WHERE window_start >= TIMESTAMP_TRUNC('$BURST_AT', MINUTE)"

In [ ]:
q "SELECT window_start, store_id, sales, revenue, pane FROM \`$PROJECT.$DATASET.sales_baseline\`
   WHERE window_start >= TIMESTAMP_TRUNC('$BURST_AT', MINUTE) ORDER BY sales DESC, store_id LIMIT 5"

## Step 4 · The watermark, in the console, from the step 2 links

## Step 5 · A sale from 30 minutes ago

In [ ]:
LATE_PUB=$(date -u +%Y-%m-%dT%H:%M:%SZ)
(cd pipeline && $PY registers.py "$T/$TOPIC" late)

In [ ]:
until [ "$(n "SELECT COUNT(*) FROM \`$PROJECT.$DATASET.sales_lateness\` WHERE store_id = 'CLT-031' AND window_start BETWEEN TIMESTAMP_SUB(TIMESTAMP('$LATE_PUB'), INTERVAL 31 MINUTE) AND TIMESTAMP_SUB(TIMESTAMP('$LATE_PUB'), INTERVAL 29 MINUTE)")" -ge 1 ]; do
  sleep 15
done
q "SELECT window_start, store_id, sales, revenue, pane FROM \`$PROJECT.$DATASET.sales_baseline\`
   WHERE store_id = 'CLT-031' AND window_start BETWEEN TIMESTAMP_SUB(TIMESTAMP('$LATE_PUB'), INTERVAL 31 MINUTE) AND TIMESTAMP_SUB(TIMESTAMP('$LATE_PUB'), INTERVAL 29 MINUTE) ORDER BY window_start"
echo "baseline rows above: none means the late sale was dropped"

## Step 6 · The allowed-lateness job re-fires the window

In [ ]:
q "SELECT window_start, store_id, sales, revenue, pane FROM \`$PROJECT.$DATASET.sales_lateness\`
   WHERE store_id = 'CLT-031' AND window_start BETWEEN TIMESTAMP_SUB(TIMESTAMP('$LATE_PUB'), INTERVAL 31 MINUTE) AND TIMESTAMP_SUB(TIMESTAMP('$LATE_PUB'), INTERVAL 29 MINUTE) ORDER BY window_start"

## Step 7 · One sale, published twice

In [ ]:
DUP_AT=$(date -u +%Y-%m-%dT%H:%M:%SZ)
(cd pipeline && $PY registers.py "$T/$TOPIC" duplicate)

In [ ]:
until [ "$(n "SELECT SUM(sales) FROM \`$PROJECT.$DATASET.sales_baseline\` WHERE store_id='CLT-007' AND window_start >= TIMESTAMP_TRUNC('$DUP_AT', MINUTE)")" -ge 2 ] \
   && [ "$(n "SELECT SUM(sales) FROM \`$PROJECT.$DATASET.sales_dedup\` WHERE store_id='CLT-007' AND window_start >= TIMESTAMP_TRUNC('$DUP_AT', MINUTE)")" -ge 1 ]; do
  sleep 15
done
q "SELECT 'baseline' job, SUM(sales) sales, ROUND(SUM(revenue),2) revenue FROM \`$PROJECT.$DATASET.sales_baseline\` WHERE store_id='CLT-007' AND window_start >= TIMESTAMP_TRUNC('$DUP_AT', MINUTE)
   UNION ALL SELECT 'dedup', SUM(sales), ROUND(SUM(revenue),2) FROM \`$PROJECT.$DATASET.sales_dedup\` WHERE store_id='CLT-007' AND window_start >= TIMESTAMP_TRUNC('$DUP_AT', MINUTE)"

## Step 8 · A malformed record

In [ ]:
(cd pipeline && $PY registers.py "$T/$TOPIC" malformed)

In [ ]:
until OUT=$(gcloud pubsub subscriptions pull "dlq-$SUFFIX" --project "$PROJECT" --auto-ack --limit 5 \
        --format="yaml(message.data.decode(base64),message.attributes.error,message.attributes.sale_id)") \
      && [ -n "$OUT" ]; do
  sleep 15
done
echo "$OUT"

In [ ]:
gcloud pubsub subscriptions pull "dlq-$SUFFIX" --project "$PROJECT" --auto-ack --limit 5 \
  --format="yaml(message.data.decode(base64),message.attributes.error,message.attributes.sale_id)"

## Step 9 · Lag, backlog, and three jobs still running

In [ ]:
gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name~crown-.*-$SUFFIX" \
  --format="table(name,state)"

## Step 10 · Drain one, cancel another

In [ ]:
BID=$(gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name=crown-baseline-$SUFFIX" --format="value(id)")
LID=$(gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name=crown-lateness-$SUFFIX" --format="value(id)")
echo "baseline $BID"; echo "lateness $LID"

In [ ]:
gcloud dataflow jobs drain "${BID:?}" --project "$PROJECT" --region "$REGION"
gcloud dataflow jobs cancel "${LID:?}" --project "$PROJECT" --region "$REGION"

In [ ]:
until gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --filter="name=crown-baseline-$SUFFIX AND state=Drained" --format="value(id)" 2>/dev/null | grep -q .; do
  sleep 15
done
gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --filter="name~crown-.*-$SUFFIX" \
  --format="table(name,state)"

## Step 11 · Teardown

In [ ]:
for j in $(gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name~crown-.*-$SUFFIX" --format="value(id)"); do
  gcloud dataflow jobs cancel "${j:?}" --project "$PROJECT" --region "$REGION"
done

In [ ]:
for s in baseline lateness dedup dlq; do gcloud pubsub subscriptions delete "$s-${SUFFIX:?}" --project "$PROJECT" --quiet; done
gcloud pubsub subscriptions delete "${SCRATCH:?}" --project "$PROJECT" --quiet
gcloud pubsub topics delete "${TOPIC:?}" "${DLQ:?}" "${SCRATCH:?}" --project "$PROJECT" --quiet

In [ ]:
bq --project_id="$PROJECT" rm -r -f -d "${DATASET:?}"
gcloud storage rm -r "gs://${BUCKET:?}" --project "$PROJECT" 2>&1 | tail -1

In [ ]:
gcloud dataflow jobs list --project "$PROJECT" --region "$REGION" --status=active --filter="name~crown-.*-$SUFFIX"
gcloud pubsub topics list --project "$PROJECT" --filter="name~$SUFFIX"